In [1]:
import requests
import pandas as pd
from datetime import date
from pathlib import Path
import time

PROJECT_ROOT = Path("..").resolve()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

SNAPSHOT = date.today().strftime("%Y%m%d")

print(f"Raw layer:     {RAW_DIR}")
print(f"Snapshot date: {SNAPSHOT}")

Raw layer:     C:\Users\Fatima zahra\Projects\CMS-hospital-typology\data\raw
Snapshot date: 20260520


In [7]:
DATASETS = {
    "hospital_general_info":   "xubh-q36u",
    "readmissions_and_deaths": "ynj2-r877",
    "medicare_spending":       "nrth-mfg3",
    "healthcare_infections":   "77hc-ibv8",
    "hcahps_patient_survey":   "dgck-syfz",
    "unplanned_visits":        "632h-zaca",
    "timely_effective_care":   "yv7e-xc69",
    ##"payment_and_value":       "c7us-v4mf", CMS is probably have it in EXCEL format , -- download later if needed 
}

In [8]:
CMS_METASTORE = "https://data.cms.gov/provider-data/api/1/metastore/schemas/dataset/items"

def get_csv_url(dataset_id: str) -> str:
    """Look up the latest CSV download URL from the dataset's metadata."""
    r = requests.get(f"{CMS_METASTORE}/{dataset_id}", timeout=30)
    r.raise_for_status()
    meta = r.json()
    for dist in meta.get("distribution", []):
        url = dist.get("downloadURL", "")
        if url.lower().endswith(".csv"):
            return url
    raise ValueError(f"No CSV distribution found for {dataset_id}")


def fetch_dataset(dataset_id: str, max_retries: int = 3) -> pd.DataFrame:
    """Fetch a CMS dataset by downloading its published CSV directly."""
    csv_url = get_csv_url(dataset_id)

    for attempt in range(max_retries):
        try:
            return pd.read_csv(csv_url, low_memory=False)
        except Exception as e:
            if attempt == max_retries - 1:
                raise
            wait = 2 ** attempt
            print(f"  attempt {attempt+1} failed ({e}), retrying in {wait}s...")
            time.sleep(wait)

In [9]:
results = {}

for name, dataset_id in DATASETS.items():
    out_path = RAW_DIR / f"{name}_{SNAPSHOT}.csv"

    # Snapshot-dating means re-runs on the same day are no-ops by default.
    # Delete the file (or change SNAPSHOT) if you want to force a re-pull.
    if out_path.exists():
        print(f"{name:<28} already pulled today — skipping")
        results[name] = "skipped"
        continue

    print(f" {name:<28} ({dataset_id})  fetching...")
    try:
        df = fetch_dataset(dataset_id)
        df.to_csv(out_path, index=False)
        print(f" {name:<28} {len(df):>7,} rows → {out_path.name}")
        results[name] = len(df)
    except Exception as e:
        print(f" {name:<28} FAILED: {e}")
        results[name] = f"error: {e}"

print("\n--- Summary ---")
for name, status in results.items():
    print(f"  {name:<28} {status}")

hospital_general_info        already pulled today — skipping
readmissions_and_deaths      already pulled today — skipping
medicare_spending            already pulled today — skipping
healthcare_infections        already pulled today — skipping
hcahps_patient_survey        already pulled today — skipping
unplanned_visits             already pulled today — skipping
timely_effective_care        already pulled today — skipping

--- Summary ---
  hospital_general_info        skipped
  readmissions_and_deaths      skipped
  medicare_spending            skipped
  healthcare_infections        skipped
  hcahps_patient_survey        skipped
  unplanned_visits             skipped
  timely_effective_care        skipped
